In [1]:
import glob
import pandas as pd
import numpy as np
import scipy.stats
import matplotlib.pyplot as plt
# import seaborn as sns

import h5py

import os.path as op
import os
from os.path import join, exists
import glob
import shutil
from datetime import datetime
import math

# from nilearn import plotting
# from nilearn import image
# from nilearn import masking
# import nilearn

import nibabel as nib

import scipy

import csv

from tqdm import tqdm

In [26]:
datestring = datetime.now()
print(datestring)
timestampStr = datestring.strftime("%b%d_%Y")
print(timestampStr)

2026-03-31 18:01:53.363890
Mar31_2026


The goal here should be to obtain betas for each emotion word for each voxel in the language localizer binary mask. I'm not entirely sure what export format would be best for this data type -- I don't know if it would be too small to fit a csv or if we should keep .nii or if we should use .npy or something.

The next step in data analysis is to compute an RSA matrix and then to be able to compare it to the behavioral matrix.

In [41]:
##define variables

sub_uid='sub-004' #CHANGE PER SUB

task_name= 'emotion_word' 

glmsingle_model = 'TYPED_FITHRF_GLMDENOISE_RR.hdf5'
exp_num_words = 24
exp_num_trials = 72
# exp_num_trs = 196

localizer_id = 'langlocSN'
# rois = {1:'LH_IFGorb', 2:'LH_IFG', 3:'LH_MFG', 4:'LH_AntTemp', 5:'LH_PostTemp', 6:'LH_AnG', 7:'RH_IFGorb', 8:'RH_IFG', 9:'RH_MFG', 10:'RH_AntTemp', 11:'RH_PostTemp', 12:'RH_AnG', 13: 'ALL'}
rois = {1: 'ALL'}

In [42]:
##set paths 
# TO CHANGE: change all paths to correct paths (ask where each thing is) - DONE

# set path to stimsets and stimset name
path_to_stimsets = '/usr/people/bs1799/neu502b/neu502b_fmri/emotion_word_glmsingle/emotion_word_stim'
# path_to_stimsets = '/Users/bianca/Desktop/NEU502B/neu502b_fmri/emotion_word_glmsingle/emotion_word_stim'
stimset_name = f'stimset_emotion_word_{sub_uid}.csv'
# path_to_stimsets = '/nese/mit/group/evlab/u/holson/EXPT_LBLLM/analyses/glmsingle/stimset_outputs/lbllm'
# stimset_name = f'stimset_lbllm_{sub_uid}_all_sessions.csv'

#path to design matrices
path_to_design_matrices = '/usr/people/bs1799/neu502b/neu502b_fmri/emotion_word_glmsingle/design_matrices'
# '/nese/mit/group/evlab/u/holson/EXPT_LBLLM/analyses/glmsingle/design_matrices/lbllm'

#path to fROI masks
path_to_fROI_mask = f'/usr/people/zt4569/neu502b/neu502b_fmri/localizer_masks/{sub_uid}'
# path_to_fROI_mask= f'/nese/mit/group/evlab/u/holson/LBLLM_analysis/output_localizers/langloc/top10percent/{sub_uid}_*_PL2017'

#path to glmsingle output
if sub_uid == "sub-004":
    path_to_glmsingle_output = f'/usr/people/bs1799/neu502b/neu502b_fmri/emotion_word_glmsingle/output_glmsingle/output_glmsingle_UID-{sub_uid}_pcstop-5_fracs-0.05_brainR2-0.7727'
else:
    path_to_glmsingle_output = f'/usr/people/bs1799/neu502b/neu502b_fmri/emotion_word_glmsingle/output_glmsingle/output_glmsingle_UID-{sub_uid}_pcstop-5_fracs-0.05'
# path_to_glmsingle_output= f'/nese/mit/group/evlab/u/holson/LBLLM_analysis/output_glmsingle/output_glmsingle/output_glmsingle_preproc-swr_pcstop5_fracs-0.05_UID-{sub_uid}'

#path to outputs
path_to_outputs = f'/usr/people/bs1799/neu502b/neu502b_fmri/emotion_word_glmsingle/output_betas/{sub_uid}'
# path_to_outputs= f'/nese/mit/group/evlab/u/holson/LBLLM_analysis/output_glmsingle/outputs_{timestampStr}/{sub_uid}'

In [43]:
## get order of itemIDs from stimset 
stimset_table = pd.read_csv(join(path_to_stimsets, stimset_name))

all_word_ids_in_order = list(map(str, list(map(int, list(stimset_table["word_id"])))))
word_ids = list(map(str, [i for i in range(1, 25)]))

# # open the design matrix - list of np arrays, one for each run 
# design_matrix = pd.read_pickle(join(path_to_design_matrices, design_matrix_name))

# # dictionary where each run is a key (by run_idx) and each value is an ordered list of the conditions in the run (by item_id)
# item_id_run_dict = {}
# # list of all item_ids in chronological order (should match glm_single output)
# all_item_ids_in_order = []

# for i, run_design_matrix in enumerate(design_matrix):
#     run_idx = i + 1 
    
#     item_id_run_dict[str(run_idx)] = []

#     # get shape of design matrix and check that it matches what we expect  
#     num_trs, num_conds = run_design_matrix.shape
#     assert num_trs == exp_num_trs, "unexpected number of time points in design matrix"
#     assert num_conds == exp_num_trials, "unexpected number of conditions in design matrix"

#     # for each tr in the design matrix, record the item id of the corresponding condition if there is a 1
#     for i, design_row in enumerate(run_design_matrix):
#         if np.sum(design_row) != 0:
#             item_id = str(np.argwhere(design_row != 0)[0, 0] + 1)
#             item_id_run_dict[str(run_idx)].append(item_id)
#             all_item_ids_in_order.append(item_id)

print(all_word_ids_in_order)
print(word_ids)

['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '10', '8', '13', '16', '22', '17', '11', '15', '20', '4', '19', '23', '5', '6', '24', '21', '3', '1', '14', '12', '7', '9', '18', '2', '8', '24', '17', '14', '19', '10', '20', '6', '5', '18', '16', '15', '2', '9', '12', '3', '13', '1', '22', '21', '7', '23', '4', '11']
['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24']


In [44]:
## get word responses in language parcel voxels to individual trials

#load betas across all trials
glmsingle_model_file=h5py.File(op.join(path_to_glmsingle_output,glmsingle_model),'r')
emotion_word_betas_dataset=glmsingle_model_file['betasmd']

#check dimensions 
print(emotion_word_betas_dataset)

(x_dim, y_dim, z_dim, num_trials) = emotion_word_betas_dataset.shape

assert num_trials == exp_num_trials 

#convert to numpy array
emotion_word_trial_betas=np.zeros((x_dim, y_dim, z_dim, num_trials))
emotion_word_betas_dataset.read_direct(emotion_word_trial_betas)

# #create dictionary with raw betas for each fROI and item_id
raw_betas = {}


# TO DO:
# keep track of which presentation of the stimulus it is (1, 2, or 3) - done! 
# modify storage dictionary to keep track of stim presentation - done! 
# modify export file name to keep track of stim presentation - done! 
# add a last step that calculates average betas across all three stim presentations - done! 

#for each trial...
for trial_idx, item_id in enumerate(all_word_ids_in_order):
    
    if trial_idx < 24:
        repeat_idx = 1
        raw_betas[item_id] = {}
    elif trial_idx >= 24 and trial_idx < 48:
        repeat_idx = 2
    else:
        repeat_idx = 3

    # load trial beta map
    trial_beta_array_img = emotion_word_trial_betas[:,:,:,trial_idx]
       
    #for each fROI...    
    for roi_num, roi_label in rois.items():      

        if repeat_idx == 1:
            raw_betas[item_id][roi_label] = []
        
        # load fROI mask
        froi_mask_name = f'{roi_label}_fROI_mask_binary_top10percent.npy'
        froi_mask_img = np.load(glob.glob(join(path_to_fROI_mask,froi_mask_name))[0])
        
        # initialize output
        #output_array_item = np.zeros((x_dim, y_dim, z_dim))
        output_array_item = np.full((x_dim, y_dim, z_dim), np.nan)
        
        # find where mask is 1
        indices = np.where(froi_mask_img == 1)
        
        # get the corresponding values in trial beta map
        corresponding_values = trial_beta_array_img[indices]
        
        # assign the corresponding values to the output array at the specified indices
        output_array_item[indices] = corresponding_values
        
        # saving individual trial arrays        
        export_path = f'{path_to_outputs}/raw_betas/{roi_label}'
        export_filename = f'{export_path}/{task_name}_itemID_{item_id}_repeat_{repeat_idx}_{roi_label}_raw_betas'  
        # if no dir make dir
        if not exists(export_path):
            os.makedirs(export_path)
        # saving np array to .npy file
        np.save(f'{export_filename}.npy', output_array_item)
        
        #save non-normalized betas for each voxel in the fROI for each word repetition        
        raw_betas[item_id][roi_label].append(output_array_item)
        
        # print('raw beta mean example', raw_betas[item_id][roi_label][repeat_idx])
        


<HDF5 dataset "betasmd": shape (78, 93, 78, 72), type "<f4">


In [45]:
## get word responses in language parcel voxels averaged across repetitions 

average_item_betas = {}

# for each emotion word 
for item_id in word_ids:
    average_item_betas[item_id] = {}
    
    # for each fROI
    for roi_num, roi_label in rois.items():

        # average together voxel values for all 3 word presentations 
        raw_betas_list = raw_betas[item_id][roi_label]
        average_item_betas_array = np.nanmean(raw_betas_list, axis = 0)
        print(f"shape of average item betas array: {average_item_betas_array.shape}")

        # save numpy array in dictionary 
        average_item_betas[item_id][roi_label] = average_item_betas_array

        # saving numpy array as a file        
        export_path = f'{path_to_outputs}/average_item_betas/{roi_label}'
        export_filename = f'{export_path}/{task_name}_itemID_{item_id}_{roi_label}_average_item_betas'  
        # if no dir make dir
        if not exists(export_path):
            os.makedirs(export_path)
        # saving np array to .npy file
        np.save(f'{export_filename}.npy', average_item_betas_array)    


/tmp/ipykernel_949110/1829859781.py:14: RuntimeWarning: Mean of empty slice
  average_item_betas_array = np.nanmean(raw_betas_list, axis = 0)


shape of average item betas array: (78, 93, 78)
shape of average item betas array: (78, 93, 78)
shape of average item betas array: (78, 93, 78)
shape of average item betas array: (78, 93, 78)
shape of average item betas array: (78, 93, 78)
shape of average item betas array: (78, 93, 78)
shape of average item betas array: (78, 93, 78)
shape of average item betas array: (78, 93, 78)
shape of average item betas array: (78, 93, 78)
shape of average item betas array: (78, 93, 78)
shape of average item betas array: (78, 93, 78)
shape of average item betas array: (78, 93, 78)
shape of average item betas array: (78, 93, 78)
shape of average item betas array: (78, 93, 78)
shape of average item betas array: (78, 93, 78)
shape of average item betas array: (78, 93, 78)
shape of average item betas array: (78, 93, 78)
shape of average item betas array: (78, 93, 78)
shape of average item betas array: (78, 93, 78)
shape of average item betas array: (78, 93, 78)
shape of average item betas array: (78, 